# Probability Distributions

Companion notebook for the [Probability Distributions](https://ml-viz.vercel.app/courses/probability-statistics/02-probability-distributions) lesson.

We implement the exact math from the lesson: the PMF/PDF distinction, deriving and **empirically verifying** $\mathbb{E}[X]$ and $\mathrm{Var}(X)$ for the Bernoulli and Gaussian, checking the Gaussian normalization integral, and fitting a Bernoulli by maximum likelihood.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

BRAND, TEAL, ORANGE = '#6366f1', '#2dd4bf', '#f97316'

## 1. PMF vs PDF

A **PMF** (discrete) gives an actual probability at each value: bars that sum to 1, each $\le 1$.
A **PDF** (continuous) gives *density* (probability per unit length): the area under the curve is 1, but the curve itself can exceed 1.

We illustrate both, including a narrow Gaussian whose peak is above 1 to drive home that a density is **not** a probability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# PMF: Bernoulli(0.7) -- two bars that sum to 1
ax = axes[0]
p = 0.7
ax.bar([0, 1], [1 - p, p], width=0.25, color=[ORANGE, BRAND], alpha=0.9)
ax.set_xticks([0, 1]); ax.set_xticklabels(['0 (failure)', '1 (success)'])
ax.set_ylim(0, 1)
ax.set_title('PMF: Bernoulli(0.7)  ->  bars sum to {:.1f}'.format((1 - p) + p))
ax.set_ylabel('probability  P(X=k)')

# PDF: a narrow Gaussian whose density peak is > 1
ax = axes[1]
x = np.linspace(-1.5, 1.5, 400)
sigma = 0.1
peak = stats.norm.pdf(0, 0, sigma)
ax.plot(x, stats.norm.pdf(x, 0, sigma), color=TEAL, lw=2)
ax.axhline(1.0, color='#94a3b8', ls='--', lw=1)
ax.set_title('PDF: N(0, 0.1^2)  ->  peak density = {:.2f}  (> 1!)'.format(peak))
ax.set_ylabel('density  p(x)')

for ax in axes.flat:
    ax.grid(True, alpha=0.2)
plt.suptitle('A density is not a probability', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

# The area under the PDF still integrates to 1:
area = np.trapz(stats.norm.pdf(x, 0, sigma), x)
print('Area under the narrow-Gaussian PDF over [-1.5, 1.5]: {:.4f}'.format(area))

## 2. Key distributions at a glance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Bernoulli for several p
ax = axes[0, 0]
for pp, color in [(0.3, BRAND), (0.5, TEAL), (0.7, ORANGE)]:
    ax.bar([0, 1], [1 - pp, pp], width=0.2, alpha=0.8, color=color,
           label='p={}'.format(pp), align='center')
ax.set_xticks([0, 1]); ax.set_xticklabels(['0 (failure)', '1 (success)'])
ax.set_title('Bernoulli PMF'); ax.legend()

# Gaussian for several (mu, sigma)
ax = axes[0, 1]
x = np.linspace(-6, 10, 400)
for mu, sig, color in [(0, 1, BRAND), (3, 1.5, ORANGE), (0, 2, TEAL)]:
    ax.plot(x, stats.norm.pdf(x, mu, sig), color=color, lw=2,
            label='N({}, {}^2)'.format(mu, sig))
ax.set_title('Gaussian PDF'); ax.legend()

# Central Limit Theorem
rng = np.random.default_rng(42)
ax = axes[1, 0]
for n, color in [(1, BRAND), (5, ORANGE), (30, TEAL)]:
    means = rng.uniform(0, 1, size=(5000, n)).mean(axis=1)
    ax.hist(means, bins=50, alpha=0.5, density=True, color=color, label='n={}'.format(n))
ax.set_title('Central Limit Theorem\n(mean of n Uniform[0,1] samples)')
ax.legend()

# 2D Gaussian contours
ax = axes[1, 1]
grid = np.linspace(-3, 3, 120)
XX, YY = np.meshgrid(grid, grid)
xy = np.column_stack([XX.ravel(), YY.ravel()])
for mu, cov, color, label in [
    ([0, 0], [[1, 0.8], [0.8, 1]], BRAND, 'rho=0.8'),
    ([0, 0], [[1, -0.5], [-0.5, 1]], ORANGE, 'rho=-0.5'),
]:
    density = stats.multivariate_normal(mu, cov).pdf(xy).reshape(120, 120)
    ax.contour(grid, grid, density, levels=5, colors=[color], alpha=0.7)
    ax.plot([], [], color=color, label=label)
ax.set_aspect('equal'); ax.set_title('2D Gaussian: different covariances')
ax.legend()

for ax in axes.flat:
    ax.grid(True, alpha=0.2)
plt.suptitle('Key Distributions in Machine Learning', y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

## 3. Verifying E[X] and Var(X) against the closed forms

The lesson derives, from first principles:

- **Bernoulli(p):** $\mathbb{E}[X] = p$, $\mathrm{Var}(X) = p(1-p)$.
- **Gaussian($\mu,\sigma^2$):** $\mathbb{E}[X] = \mu$, $\mathrm{Var}(X) = \sigma^2$.
- **Uniform[a,b]:** $\mathbb{E}[X] = \tfrac{a+b}{2}$, $\mathrm{Var}(X) = \tfrac{(b-a)^2}{12}$.

Now sample heavily and confirm the empirical moments match.

In [ ]:
rng = np.random.default_rng(0)
N = 200000

distributions = [
    ('Bernoulli(0.7)', rng.binomial(1, 0.7, N), 0.7, 0.7 * 0.3),
    ('Gaussian(2, 4)', rng.normal(2, 2, N),      2.0, 4.0),       # sigma=2 -> var=4
    ('Uniform[0, 1]',  rng.uniform(0, 1, N),     0.5, 1 / 12),
]

header = '{:18s}  {:>11s}  {:>11s}  {:>11s}  {:>11s}'.format(
    'Distribution', 'E[X] true', 'E[X] samp', 'Var true', 'Var samp')
print(header)
print('-' * len(header))
for name, s, true_mean, true_var in distributions:
    print('{:18s}  {:11.4f}  {:11.4f}  {:11.4f}  {:11.4f}'.format(
        name, true_mean, s.mean(), true_var, s.var()))

### Var(X) = E[X^2] - (E[X])^2, checked numerically

The lesson's shortcut for variance. We confirm both sides agree on a Gaussian sample.

In [ ]:
s = rng.normal(2, 2, N)
lhs = s.var()                       # E[(X-mu)^2]
rhs = (s ** 2).mean() - s.mean() ** 2  # E[X^2] - (E[X])^2
print('E[(X-mu)^2]            = {:.4f}'.format(lhs))
print('E[X^2] - (E[X])^2      = {:.4f}'.format(rhs))
print('true sigma^2           = {:.4f}'.format(4.0))

## 4. The Gaussian normalization integral

The lesson shows $\int_{-\infty}^{\infty} e^{-x^2/2}\,dx = \sqrt{2\pi}$ via the polar-coordinate trick, so the full PDF (divided by $\sigma\sqrt{2\pi}$) integrates to 1. We verify both numerically.

In [ ]:
# Bare bell curve should integrate to sqrt(2*pi)
x = np.linspace(-12, 12, 100001)
bare = np.exp(-x ** 2 / 2)
print('integral of e^(-x^2/2)   = {:.6f}'.format(np.trapz(bare, x)))
print('sqrt(2*pi)               = {:.6f}'.format(np.sqrt(2 * np.pi)))

# Full normalized PDF integrates to 1 for any (mu, sigma)
for mu, sigma in [(0, 1), (3, 1.5), (-2, 0.5)]:
    xx = np.linspace(mu - 12 * sigma, mu + 12 * sigma, 100001)
    pdf = np.exp(-(xx - mu) ** 2 / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))
    print('N({:>4}, {:>4}) total area = {:.6f}'.format(mu, sigma, np.trapz(pdf, xx)))

## 5. Fitting a Bernoulli by maximum likelihood

The lesson derives the MLE for a coin: with $s$ heads in $n$ flips, the log-likelihood
$\ell(\theta) = s\log\theta + (n-s)\log(1-\theta)$ is maximized at $\hat\theta = s/n$.

We plot $\ell(\theta)$ for simulated data and confirm its peak lands on the head-fraction.

In [ ]:
rng = np.random.default_rng(7)
true_p = 0.7
n = 200
flips = rng.binomial(1, true_p, n)
s = flips.sum()

theta = np.linspace(0.001, 0.999, 999)
loglik = s * np.log(theta) + (n - s) * np.log(1 - theta)

mle_closed = s / n                 # derived in the lesson
mle_grid = theta[np.argmax(loglik)]  # numerical argmax for a sanity check

print('heads s = {} of n = {}'.format(s, n))
print('closed-form MLE  s/n   = {:.4f}'.format(mle_closed))
print('grid-search argmax     = {:.4f}'.format(mle_grid))
print('true p                 = {:.4f}'.format(true_p))

plt.figure(figsize=(8, 4.5))
plt.plot(theta, loglik, color=BRAND, lw=2)
plt.axvline(mle_closed, color=TEAL, ls='--', lw=1.5,
            label='MLE = s/n = {:.3f}'.format(mle_closed))
plt.axvline(true_p, color=ORANGE, ls=':', lw=1.5,
            label='true p = {:.2f}'.format(true_p))
plt.xlabel('theta'); plt.ylabel('log-likelihood  l(theta)')
plt.title('Bernoulli log-likelihood peaks at the head-fraction')
plt.grid(True, alpha=0.2); plt.legend()
plt.tight_layout(); plt.show()